# Week 1 — Interactive: predictability of the Lorenz 63 systemThis notebook runs **entirely in your browser** — no installation, no account, nothing to download.Click a cell and press **Shift+Enter** to run it. Edit anything you like; you cannot break the course website.It reproduces the two key figures of Week 1 and demonstrates the central claim:**predictability is a property of the state, not a single universal number.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. The modelThe Lorenz 63 system, exactly as in equation (eq2) of the notes:$$\frac{dx}{dt}=\sigma(y-x),\qquad  \frac{dy}{dt}=x(\rho-z)-y,\qquad  \frac{dz}{dt}=xy-\beta z$$The state is stored as an array of shape `(3, N)` so that `N` trajectories are advanced at once —that vectorisation is what makes this fast enough to run in a browser.

In [ ]:
SIGMA, RHO, BETA = 10.0, 28.0, 8.0/3.0   # the classic chaotic parameter set

def lorenz(Y, sigma=SIGMA, rho=RHO, beta=BETA):
    """Right-hand side of L63. Y has shape (3, N); returns the same shape."""
    x, y, z = Y
    return np.stack([sigma*(y - x),
                     x*(rho - z) - y,     # note the -y term
                     x*y - beta*z])

def rk4_step(Y, dt, **kw):
    k1 = lorenz(Y, **kw)
    k2 = lorenz(Y + dt*k1/2, **kw)
    k3 = lorenz(Y + dt*k2/2, **kw)
    k4 = lorenz(Y + dt*k3,   **kw)
    return Y + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def integrate(Y0, nsteps, dt, keep=True, **kw):
    """Advance nsteps. keep=True returns the whole trajectory (nsteps+1, 3, N)."""
    Y = Y0.copy()
    if not keep:
        for _ in range(nsteps):
            Y = rk4_step(Y, dt, **kw)
        return Y
    out = np.empty((nsteps+1,) + Y0.shape)
    out[0] = Y
    for i in range(nsteps):
        Y = rk4_step(Y, dt, **kw)
        out[i+1] = Y
    return out

dt = 0.01

## 2. The climatological distributionRecall the definition: the **climatological distribution** is what you are left with once the forecastcarries no information — i.e. the long-term distribution of the system.Rather than one very long run, we exploit ergodicity and run 200 trajectories in parallel after aspin-up onto the attractor. Statistically this is equivalent, and much faster.

In [ ]:
Y0   = rng.normal(size=(3, 200))*5 + np.array([0., 0., 25.])[:, None]
Y0   = integrate(Y0, 2000, dt, keep=False)      # spin-up: land on the attractor
clim = integrate(Y0, 1000, dt)                  # (1001, 3, 200)

pool = clim.transpose(1, 0, 2).reshape(3, -1)   # (3, 200200) samples of the attractor
print(f"{pool.shape[1]:,} climatological samples;  std(x) = {pool[0].std():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(clim[:, 0, 0], clim[:, 2, 0], lw=0.4, color='gray')
ax.set_xlabel('x'); ax.set_ylabel('z'); ax.set_title('The Lorenz attractor (one trajectory)')
plt.show()

## 3. Predictability depends on where you startNow the central experiment. We pick 300 states at random *from the attractor*, and around each one welaunch a 50-member ensemble whose members differ only by a tiny perturbation (0.01).We integrate every ensemble forward by the same lead time, and measure the resulting spread.If predictability were a universal number, every ensemble would spread by the same amount.

In [ ]:
M, LEAD, EPS = 50, 200, 0.01      # members, lead steps (=2 time units), perturbation size

idx    = rng.choice(pool.shape[1], 300, replace=False)
launch = pool[:, idx]                                        # (3, 300) starting states
ens0   = launch[:, None, :] + EPS*rng.normal(size=(3, M, 300))
traj   = integrate(ens0.reshape(3, -1), LEAD, dt).reshape(LEAD+1, 3, M, 300)

spread = traj[-1, 0].std(axis=0)          # ensemble std of x at the final lead, per launch state
order  = np.argsort(spread)
slow, mid, fast = order[0], order[len(order)//2], order[-1]

print(f"final ensemble spread in x:  slowest {spread[slow]:.3f}   "
      f"median {spread[mid]:.3f}   fastest {spread[fast]:.3f}")
print(f"climatological spread in x:  {pool[0].std():.3f}")

Look at those three numbers against the climatological spread. The fastest-growing case has alreadyreached climatological spread — its forecast is worthless. The slowest is still essentially a point.**Same model, same perturbation size, same lead time.**### Reproducing FIG2 — the three error-growth regimes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True, sharey=True)
for ax, k, lab in zip(axes, [fast, mid, slow], ['(a) fast', '(b) average', '(c) slow']):
    ax.plot(clim[:, 0, 0], clim[:, 2, 0], lw=0.3, color='lightgray', zorder=0)
    for m in range(M):
        ax.plot(traj[:, 0, m, k], traj[:, 2, m, k], lw=0.7, alpha=0.7)
    ax.plot(launch[0, k], launch[2, k], 'k*', ms=13, zorder=5)
    ax.set_title(f'{lab} error growth  (spread = {spread[k]:.2f})')
    ax.set_xlabel('x')
axes[0].set_ylabel('z')
fig.suptitle('Ensemble forecasts launched from three different states (star = initial state)')
plt.tight_layout(); plt.show()

### Reproducing FIG3 — forecast PDF vs climatological PDFThis is the statistical test from the notes made visual. The predictability limit is reached when theforecast PDF becomes indistinguishable from the climatological PDF.

In [ ]:
bins = np.linspace(-25, 25, 60)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, k, lab in zip(axes, [fast, mid, slow], ['(a) fast', '(b) average', '(c) slow']):
    ax.hist(pool[0], bins=bins, density=True, alpha=0.45,
            color='gray', label='Climatological PDF')
    ax.hist(traj[-1, 0, :, k], bins=bins, density=True, alpha=0.65,
            color='crimson', label='Forecast PDF')
    ax.set_title(lab); ax.set_xlabel('x')
axes[0].set_ylabel('density'); axes[0].legend()
fig.suptitle('Forecast PDF at the final lead time, against the climatological PDF')
plt.tight_layout(); plt.show()

In (a) the forecast PDF has spread across the whole attractor and is close to the climatological one —we can no longer reject the null hypothesis of eq (eq1), so the **predictability limit has been reached**.In (b) and (c) the forecast PDF is still a narrow spike, clearly distinguishable from climatology — thoseforecasts still carry information.## 4. Things to try1. **Change the lead time.** Set `LEAD = 50, 100, 400` and re-run from section 3. At what lead does the   *median* case become unpredictable? That number is your "rule of thumb" predictability limit — and   notice how poorly it describes the individual cases.2. **Change the perturbation size.** Try `EPS = 1e-4` and `EPS = 1.0`. A hundred-fold reduction in initial   error buys you surprisingly little extra lead time. This is the practical meaning of the statement in   the notes that *the predictability limit is inevitable*.3. **Change the parameters.** Try `RHO = 15` (below the onset of chaos) and re-run everything. What happens   to the spread between the fast and slow cases, and why?4. **Where are the unpredictable states?** Plot `launch[0]` against `launch[2]`, coloured by `spread`, to   see *where on the attractor* forecasts go bad. Do you recognise the region?